In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

import warnings
warnings.filterwarnings("ignore")

In [7]:
db_path = "db.sqlite3"

In [8]:
with sqlite3.connect(db_path) as conn:
    tables = pd.read_sql_query(
        """
        SELECT name AS table_name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name;
        """,
        conn
    )

display(tables)

,table_name
0,auth_group
1,auth_group_permissions
2,auth_permission
3,auth_user
4,auth_user_groups
5,auth_user_user_permissions
6,django_admin_log
7,django_content_type
8,django_migrations
9,django_session


In [9]:
table_name = "mockat_target"

In [10]:
with sqlite3.connect(db_path) as conn:
    df = pd.read_sql_query(
        f'SELECT * FROM "{table_name}"',
        conn
    )

display(df)

,id,name,pRA,pdec,mag1,mag2,mag0,group,notes,infield,nearby,prior,mag3
0,1,None,97.072592,-13.053056,7.86,0.0,8.52,fake:,"{""type"": ""none"", ""eclipt_bl"": [98, -36], ""mag_...",,,0.0,0.0
1,2,HD 50138,102.889162,-6.966514,6.28,0.0,6.68,merg:,"{""type"": ""none"", ""eclipt_bl"": [104, -29], ""mag...",,,2.0,0.0
2,3,AS 174,116.872082,-31.844533,12.33,0.0,12.41,merg:,"{""type"": ""none"", ""eclipt_bl"": [128, -51], ""mag...",,,0.0,0.0
3,4,AS 225,260.231405,-38.000102,0.00,0.0,14.82,merg:,"{""type"": ""none"", ""eclipt_bl"": [262, -14], ""mag...",,,0.0,0.0
4,5,StHa 145,266.763643,23.329265,0.00,0.0,10.04,merg:,"{""type"": ""none"", ""eclipt_bl"": [265, 46], ""mag_...",,,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
424,540,EP_01709278114,311.450000,14.142000,999.00,999.0,999.00,,"{""xray"": 8.9, ""ra_dec_error"": 0.05087978, ""tri...",,,0.0,999.0
425,541,EP_01709278015,353.598000,-15.513000,999.00,999.0,999.00,,"{""xray"": 7.6, ""ra_dec_error"": 0.05136735, ""tri...",,,0.0,999.0
426,542,EP_01709277768,328.573000,17.607000,999.00,999.0,999.00,,"{""xray"": 8.2, ""ra_dec_error"": 0.05105352, ""tri...",,,0.0,999.0
427,543,EP_01709277716,12.278000,35.588000,999.00,999.0,999.00,,"{""xray"": 7.0, ""ra_dec_error"": 0.05183406, ""tri...",,,0.0,999.0


In [ ]:
with sqlite3.connect(db_path) as conn:
    groups = pd.read_sql_query(
        """
        SELECT "group", COUNT(*) AS n_targets
        FROM mockat_target
        GROUP BY "group"
        ORDER BY n_targets DESC
        """,
        conn
    )

display(groups)

In [ ]:
def set_target_prior(prior, group=None, target_ids=None):
    """Set mockat_target.prior for selected targets. Pass exactly one of:

    group: prefix match on the "group" column, e.g. group='agn' matches
           'agn:' (groups are stored like 'agn:', 'merg:', 'pulsar:FRB' --
           see the groups cell above for the exact values).
    target_ids: a single Target id or a list of Target ids (the 'id'
           column in mockat_target, not an Objective id).
    """
    if (group is None) == (target_ids is None):
        raise ValueError("pass exactly one of group or target_ids")
    with sqlite3.connect(db_path) as conn:
        cur = conn.cursor()
        if group is not None:
            cur.execute('UPDATE mockat_target SET prior = ? WHERE "group" LIKE ?',
                        (prior, group + '%'))
        else:
            if isinstance(target_ids, int):
                target_ids = [target_ids]
            cur.executemany('UPDATE mockat_target SET prior = ? WHERE id = ?',
                             [(prior, tid) for tid in target_ids])
        conn.commit()
        print(f"updated {cur.rowcount} target(s)")

# examples:
# set_target_prior(2, group='agn')          # all AGN targets -> prior 2
# set_target_prior(1.5, target_ids=[12, 34])

set_target_prior(2, group='agn')

In [11]:
import json

# schedme is a per-Objective tag (mockat_objective.notes['camp']), not a
# Target field -- print each objective's id, its target's name, and its
# full notes, flagging whether it's currently in the schedme campaign.
with sqlite3.connect(db_path) as conn:
    obj_df = pd.read_sql_query(
        """
        SELECT o.id, o.target_id, t.name AS target_name, o.notes, o.scheduled_line
        FROM mockat_objective o
        LEFT JOIN mockat_target t ON t.id = o.target_id
        ORDER BY o.id
        """,
        conn
    )

for _, row in obj_df.iterrows():
    notes = json.loads(row['notes'])
    camp = notes.get('camp')
    flag = 'SCHEDME' if camp and str(camp).startswith('schedme') else '       '
    print(f"[{flag}] obj {row['id']:>4}  target {row['target_id']:>4} {row['target_name'] or '':<20}  notes={notes}")

[       ] obj    1  target    5 StHa 145              notes={'type': 'none'}
[       ] obj    2  target    6 MWC 342               notes={'type': 'none'}
[       ] obj    3  target    6 MWC 342               notes={'type': 'none'}
[       ] obj    4  target    1 None                  notes={'type': 'none'}
[       ] obj    5  target    2 HD 50138              notes={'type': 'none'}
[       ] obj    6  target    1 None                  notes={'type': 'none'}
[       ] obj    7  target    2 HD 50138              notes={'type': 'none'}
[       ] obj    8  target    3 AS 174                notes={'type': 'none'}
[       ] obj    9  target    4 AS 225                notes={'type': 'none'}
[       ] obj   10  target    7 MWC 349               notes={'type': 'none'}
[       ] obj   11  target    1 None                  notes={'type': 'none'}
[       ] obj   12  target    8 FBS0022-021           notes={'type': 'none'}
[       ] obj   13  target    9 IRAS 00470+6429       notes={'type': 'none'}

In [ ]:
def set_schedme(obj_ids, add=True, camp_value='schedme'):
    """Add or remove the schedme campaign tag on mockat_objective row(s).

    obj_ids: a single objective id or a list of ids (these are Objective
             ids, i.e. the 'id' column printed above -- not target ids).
    add=True: sets notes['camp'] = camp_value (default 'schedme'; pass e.g.
             'schedme_trans' for the transient sub-campaign).
    add=False: removes the 'camp' key entirely.
    """
    if isinstance(obj_ids, int):
        obj_ids = [obj_ids]
    with sqlite3.connect(db_path) as conn:
        cur = conn.cursor()
        for oid in obj_ids:
            cur.execute('SELECT notes FROM mockat_objective WHERE id = ?', (oid,))
            row = cur.fetchone()
            if row is None:
                print(f"objective {oid} not found")
                continue
            notes = json.loads(row[0])
            if add:
                notes['camp'] = camp_value
            else:
                notes.pop('camp', None)
            cur.execute('UPDATE mockat_objective SET notes = ? WHERE id = ?',
                        (json.dumps(notes), oid))
        conn.commit()
    print(f"updated {len(obj_ids)} objective(s)")

# examples:
# set_schedme([12, 34], add=True)     # put objectives 12 and 34 into schedme
# set_schedme(56, add=False)          # take objective 56 out of schedme

In [ ]:
def create_objective(target_id, expo, repeat=1, delay=0, camp='schedme'):
    """Create a brand-new mockat_objective row for a target that doesn't
    have one yet (set_schedme only edits EXISTING rows -- it can't tag
    a target that has no Objective at all).

    target_id: the Target's id (the one in mockat_target, not its name).
    expo: exposure per repeat, in seconds.
    """
    notes = json.dumps({'camp': camp})
    with sqlite3.connect(db_path) as conn:
        cur = conn.cursor()
        cur.execute(
            'INSERT INTO mockat_objective (expo, notes, target_id, delay, repeat, scheduled_line) '
            'VALUES (?, ?, ?, ?, ?, NULL)',
            (expo, notes, target_id, delay, repeat)
        )
        conn.commit()
        print(f"created objective {cur.lastrowid} for target {target_id} (camp={camp})")

# Magnetar targets currently have no Objective at all -- create one for each
# so they can be picked up by sched_first.py (adjust expo to what you need):
# magnetar_ids = [71, 67, 75, 72]   # 1E 1547.0-5408, PSR J0100-7211, SGR 0501+451, SGR 0526-66
# for tid in magnetar_ids:
#     create_objective(tid, expo=200)

In [ ]:
def create_objective_by_group(group, expo, repeat=1, delay=0, camp='schedme'):
    """Like create_objective, but for every target whose group matches
    (prefix match on the "group" column, e.g. group='pulsar:Magnetar').

    Skips targets that already have an Objective -- create_objective is
    only for targets with none yet; use set_schedme / set_schedme_by_group
    to tag an existing Objective instead.
    """
    with sqlite3.connect(db_path) as conn:
        cur = conn.cursor()
        cur.execute(
            'SELECT t.id, t.name FROM mockat_target t '
            'WHERE t."group" LIKE ? '
            'AND t.id NOT IN (SELECT target_id FROM mockat_objective)',
            (group + '%',)
        )
        targets = cur.fetchall()
    if not targets:
        print(f"no targets without an existing objective found for group '{group}'")
        return
    for tid, name in targets:
        create_objective(tid, expo, repeat=repeat, delay=delay, camp=camp)

# create_objective_by_group('pulsar:Magnetar', expo=200)

In [ ]:
def set_objective_params(expo=None, repeat=None, delay=None, obj_ids=None, group=None):
    """Update expo/repeat/delay on existing mockat_objective row(s). Pass
    exactly one of:

    obj_ids: a single Objective id or a list of Objective ids (the 'id'
             column, not a target id).
    group:   prefix match on the target's "group" column, e.g. group='agn'
             matches 'agn:' -- updates every Objective belonging to a
             target in that group.

    Only the params you actually pass (not None) get changed; the rest
    are left as-is. Reminder (see _objective_total_seconds /
    _chunk_durations in orbital_scheduler.py): delay > 0 disables
    chunking entirely, forcing expo*repeat (+ delay*(repeat-1)) seconds
    to be scheduled as ONE continuous block -- fine for a short burst,
    but it must fit inside a single eclipse pass or it will never be
    schedulable. Use delay=0 (with repeat*expo as large as you like) to
    let it be split across multiple passes instead.
    """
    if (obj_ids is None) == (group is None):
        raise ValueError("pass exactly one of obj_ids or group")
    fields, values = [], []
    if expo is not None:
        fields.append('expo = ?'); values.append(expo)
    if repeat is not None:
        fields.append('repeat = ?'); values.append(repeat)
    if delay is not None:
        fields.append('delay = ?'); values.append(delay)
    if not fields:
        print("nothing to update -- pass expo/repeat/delay")
        return
    with sqlite3.connect(db_path) as conn:
        cur = conn.cursor()
        if obj_ids is not None:
            if isinstance(obj_ids, int):
                obj_ids = [obj_ids]
            cur.executemany(
                f'UPDATE mockat_objective SET {", ".join(fields)} WHERE id = ?',
                [tuple(values) + (oid,) for oid in obj_ids]
            )
        else:
            cur.execute(
                f'UPDATE mockat_objective SET {", ".join(fields)} '
                'WHERE target_id IN (SELECT id FROM mockat_target WHERE "group" LIKE ?)',
                tuple(values) + (group + '%',)
            )
        conn.commit()
        print(f"updated {cur.rowcount} objective(s)")

# examples:
# set_objective_params(expo=600, repeat=1, delay=0, group='agn')  # all AGN objectives
# set_objective_params(expo=300, obj_ids=[96, 97])                # just these two

In [ ]:
import os
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'settings')
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
import django
django.setup()
from mockat import models

def clear_sched_line(sched_line):
    """Fully clean up one schedule line, same as sched_first.py does when
    regenerating a line. Objective.objects.filter(scheduled_line=X).update(...)
    alone is NOT enough -- it would leave the actual Schedule rows (pointings)
    behind, and leave stale notes['done_by_line'] entries that double-count
    exposure the next time this line (or the objective) is scheduled.
    """
    n_sched = models.Schedule.objects.filter(line__gte=sched_line, line__lt=sched_line+1).count()
    models.Schedule.objects.filter(line__gte=sched_line, line__lt=sched_line+1).delete()

    n_obj = 0
    for obj in models.Objective.objects.filter(notes__done_by_line__has_key=str(sched_line)):
        obj.notes['done_by_line'].pop(str(sched_line), None)
        if obj.scheduled_line == sched_line:
            obj.scheduled_line = None
        obj.save(update_fields=['notes', 'scheduled_line'])
        n_obj += 1

    n_obj2 = models.Objective.objects.filter(scheduled_line=sched_line).update(scheduled_line=None)
    print(f"line {sched_line}: deleted {n_sched} Schedule rows, "
          f"cleaned {n_obj} objectives' done_by_line, cleared scheduled_line on {n_obj2} more")

# clear_sched_line(18)
# clear_sched_line(-1)

In [ ]:
conn.close()